In [1]:
import geopandas as gpd
import xarray as xr
from rasterstats import zonal_stats
import pandas as pd
import numpy as np
import rioxarray

import matplotlib.pyplot as plt

import glob
import os
from datetime import datetime

In [2]:
#change the working directory to the input folder where the netcdf files are stored
os.chdir("../")
print(os.getcwd())

c:\Users\uginet\Documents\anemie\code_archive\replication_code_JEEM_anemia-main\0_input


In [3]:
ref_date = pd.Timestamp("1900-01-01")

confinement_date = pd.Timestamp("2020-03-25")

confinement_date-ref_date

Timedelta('43913 days 00:00:00')

In [4]:
# 0. Get the coordinates of the DHS clusters
dhs_cells = pd.read_csv('./matching_grid_cells_DHS_clusters/output/Coord_cells_DHS_01.csv')
points = dhs_cells[['x', 'y']].drop_duplicates()
print(points.shape)
# 1. List NetCDF files
nc_files = sorted(glob.glob(r"./ERA5_grid_cell_data/output/010_alltemps_*.nc"))
nc_files

(17981, 2)


['./ERA5_grid_cell_data/output\\010_alltemps_201410.nc',
 './ERA5_grid_cell_data/output\\010_alltemps_201411.nc',
 './ERA5_grid_cell_data/output\\010_alltemps_201412.nc']

## The following code exctracts precipitation data at the location of every DHS cluster and compute the daily precipitation level.
### Given that the data are cumulated over each day and set back to zero at the start of each new day, getting the daily precipitation level is done by getting for each pixel, the value of the precipitations at the end of the day, ie when time == 23

In [ ]:
# Coordonnées DHS
points = pd.read_csv(
    "./matching_grid_cells_DHS_clusters/output/Coord_cells_DHS_01.csv"
)

coords = points[['x','y']].drop_duplicates()

# All the ERA5 netcdf
nc_files = sorted(
    glob.glob("./ERA5_grid_cell_data/output/010_alltemps_*.nc")
)

# Open them once
ds = xr.open_mfdataset(
    nc_files,
    combine="by_coords",
    chunks={"time": 1000}   # can be adjusted given available RAM
)

# precipitation variable
tp = ds["tp"]

# spatial selection using coords
vals = tp.sel(
    lon=xr.DataArray(coords["x"], dims="points"),
    lat=xr.DataArray(coords["y"], dims="points"),
    method="nearest"
)

# Precipations are accumulated over each day and then set back to zero at midnight. To get the daily values, get the data at 23 oclock.
vals = vals.where(vals.time.dt.hour == 23, drop=True)

# adjust to the equivalent day
times = pd.to_datetime(vals.time.values) - pd.Timedelta(days=1)

# convert to a dataframe
daily_prec = pd.DataFrame(
    vals.values.T,
    index=coords.index,
    columns=times
)

#concatenate to the coords dataframe
result = pd.concat([coords, daily_prec], axis=1)

#save
result.to_csv(
    "./Total_precip/output/daily_prec.csv",
    index=False
)

ds.close()